In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/nextgen_nlp_final")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

CLEAN_PATH = ARTIFACTS_DIR / "news_clean_relevant_v1.parquet"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("CLEAN_PATH:", CLEAN_PATH)
print("Exists?", CLEAN_PATH.exists())

PROJECT_ROOT: /content/drive/MyDrive/nextgen_nlp_final
ARTIFACTS_DIR: /content/drive/MyDrive/nextgen_nlp_final/artifacts
CLEAN_PATH: /content/drive/MyDrive/nextgen_nlp_final/artifacts/news_clean_relevant_v1.parquet
Exists? True


In [4]:
import pandas as pd

df_rel = pd.read_parquet(CLEAN_PATH)

print("Shape:", df_rel.shape)
df_rel.head()

Shape: (134546, 5)


,date,title_clean,text_clean,clean_text,clean_len
0,2024-09-22 00:00:00+00:00,"If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...","Sep 18, 2024 TL;DR: Put all your AI tools in o...",4612
1,2023-11-10 00:00:00+00:00,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...,"By Novembro 8, 2023 O camiño por diante: como ...",3621
2,2023-11-19 00:00:00+00:00,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,4290
3,2023-09-07 00:00:00+00:00,Zoom Expands AI Offering with AI Companion and...,Zoom Expands AI Offering with AI Companion and...,By Robert AndrewRelated Post Technology Cisco ...,1217
4,2023-08-04 00:00:00+00:00,Pro-AI Thinking: Enhancing Industrial Environm...,Pro-AI Thinking: Enhancing Industrial Environm...,By Mampho Brescia Related Post News Technologi...,1209


In [5]:
df_rel = df_rel.drop(columns=["text_clean", "clean_len"])

In [6]:
def build_topic_text(row, max_words=300):

    title = str(row["title_clean"])
    text = str(row["clean_text"])   # ← 这里改了

    # 只取前300词
    words = text.split()
    text_short = " ".join(words[:max_words])

    return title + ". " + text_short

In [7]:
df_rel["topic_text"] = df_rel.apply(build_topic_text, axis=1)

In [8]:
df_rel.head(3)

,date,title_clean,clean_text,topic_text
0,2024-09-22 00:00:00+00:00,"If using AI feels like a chore, try this - Boi...","Sep 18, 2024 TL;DR: Put all your AI tools in o...","If using AI feels like a chore, try this - Boi..."
1,2023-11-10 00:00:00+00:00,The Road Ahead: How China's AI Foundation Mode...,"By Novembro 8, 2023 O camiño por diante: como ...",The Road Ahead: How China's AI Foundation Mode...
2,2023-11-19 00:00:00+00:00,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...


# Stop words

In [9]:
from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer

base_stopwords = set(text.ENGLISH_STOP_WORDS)


In [10]:
extra_stopwords_v5 = {

    # news reporting language
    "new", "today", "report", "reports", "said", "using", "information",

    # pr / newswire
    "prnewswire", "newswires",

    # generic tech
    "tech", "services", "innovation", "powered",
    "mobile", "computer", "app"
}

In [11]:
custom_stopwords = base_stopwords.union({
    # AI generic
    "ai", "artificial", "intelligence", "machine", "learning",
    "model", "models", "data", "system", "systems", "technology", "technologies",
    "tool", "tools", "platform", "platforms", "software", "solution", "solutions",
    "generative", "research",

    # business / generic
    "company", "companies", "business", "industry", "industries",
    "digital", "development", "management", "future", "global", "world",
    "market", "markets", "share", "stock", "stocks", "public", "high",

    # news / website noise
    "news", "content", "home", "search", "menu", "subscribe", "watch",
    "videos", "video", "press", "updated", "latest", "live", "contact",
    "support", "help", "events", "insights", "local", "sign", "skip",
    "features", "releases", "view", "best",

    # time / numbers
    "year", "month", "day", "time", "2022", "2023", "2024", "2025", "10", "00",

    # geography / broad location noise
    "india", "china", "country", "united", "states",

    # filler / role words
    "like", "people", "work", "ceo", "human","users", "team","free","real","based","use", "today"

    # publishers / source names often dominating topics
    "prnewswire", "google", "nasdaq",

    # broad section words
    "sports", "music", "entertainment", "politics", "tv", "media"
})

In [12]:
CUSTOM_STOPWORDS = {

    # 你已经发现的
    "published","including","just","make","made","announced","announce",
    "power","email","leading","used","use","using","key","create","creates",
    "created","years","year","today",

    # 新闻文章常见废词
    "said","says","according","report","reported","reports",
    "statement","press","release","via",

    # 常见商业新闻废词
    "company","companies","business","market","industry",
    "global","world","across","major","leading",

    # 常见描述性废词
    "new","latest","next","first","top","big",
    "growing","grows","growth",

    # 时间废词
    "monday","tuesday","wednesday","thursday","friday",
    "january","february","march","april","may","june",
    "july","august","september","october","november","december",

    # 数量废词
    "million","billion","percent","per","cent",

    # 技术文章常见 filler
    "technology","tech","platform","solution","solutions",
    "system","systems","service","services","product","products",

    # marketing词
    "experience","customer","customers","capabilities","access",

    # useless verbs
    "help","helps","support","enable","enables","allow","allows",

    # generic nouns
    "thing","things","way","ways","part",

    # filler words
    "well","much","many","also","still","even"
}

In [13]:
CUSTOM_STOPWORDS2 = {

    # ===== 你刚才提到的 =====
    "am","pm","ago","day","days","year","years","today",
    "need","making","make","made",
    "open","openai","chatgpt",

    # ===== 数字相关 =====
    "one","two","three","four","five","six","seven","eight","nine","ten",
    "11","12","13","14","15","16","17","18","19","20","hours","country", "state",

    # ===== 时间表达 =====
    "monday","tuesday","wednesday","thursday","friday",
    "january","february","march","april","may","june",
    "july","august","september","october","november","december",

    # ===== 新闻 filler =====
    "said","says","according","report","reported","reports",
    "published","including","announced","announcement",

    # ===== 科技公司名 =====
    "microsoft","apple","facebook","meta","twitter","google","nvidia",

    # ===== tech generic =====
    "technology","tech","platform","platforms","system","systems",
    "product","products","applications","application",

    # ===== AI generic =====
    "ai","artificial","intelligence","model","models","language","large",

    # ===== marketing words =====
    "experience","service","services","customer","customers",

    # ===== useless verbs =====
    "use","used","using","help","helps","support","supports",
    "enable","enables","allow","allows",

    # ===== generic nouns =====
    "thing","things","way","ways","part","type","types",

    # ===== numbers from scraped text =====
    "com","www","http","https",

    # ===== random filler =====
    "well","much","many","also","still","even","ago"
}

In [14]:
custom_stopwords = custom_stopwords.union(extra_stopwords_v5)

In [15]:
custom_stopwords = custom_stopwords.union(CUSTOM_STOPWORDS)

In [16]:
custom_stopwords = custom_stopwords.union(CUSTOM_STOPWORDS2)

In [17]:
vectorizer_model = CountVectorizer(
    stop_words=list(custom_stopwords),
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.6
)

### 先sample 3000看一下效果

In [18]:
df_topic = df_rel.sample(n=30000, random_state=42).copy()
docs = df_topic["topic_text"].tolist()

print(df_topic.shape)

(30000, 4)


In [19]:
from collections import Counter

analyzer = vectorizer_model.build_analyzer()
token_counts = Counter()

for doc in docs:
    token_counts.update(analyzer(doc))

top_words_df = pd.DataFrame(token_counts.most_common(50), columns=["term", "count"])
top_words_df.head(30)

,term,count
0,cloud,7998
1,security,7093
2,potential,6530
3,health,6199
4,science,5357
5,image,5127
6,financial,5092
7,advanced,5082
8,images,5073
9,social,5056


## BERTopic_whole_data START

In [20]:
df_topic = df_rel.copy()

docs = df_topic["topic_text"].tolist()

print("Using full dataset:", df_topic.shape)
print("Number of docs:", len(docs))
print(docs[0][:500])

Using full dataset: (134546, 4)
Number of docs: 134546
If using AI feels like a chore, try this - Boing Boing. Sep 18, 2024 TL;DR: Put all your AI tools in one place with a lifetime subscription to 1minAI, now $39.99 (reg. $234). AI isn't going anywhere. The problem is that we're going everywhere trying to use it. If you want to make a blog post, you might stop by ChatGPT for the copy, head on over to midjourney for an image, and pop on by two or three other tools for editing, SEO, or maybe even making a video to earn those mixed media views. It's a


In [21]:
!pip install -q bertopic umap-learn hdbscan sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.8 MB/s eta 0:00:00


In [22]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [23]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=200,
    min_samples=20,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

In [25]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True
)

In [26]:
topics, probs = topic_model.fit_transform(docs)

2026-03-09 05:38:57,093 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/4205 [00:00<?, ?it/s]

2026-03-09 05:39:53,812 - BERTopic - Embedding - Completed ✓
2026-03-09 05:39:53,813 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-09 05:41:15,399 - BERTopic - Dimensionality - Completed ✓
2026-03-09 05:41:15,400 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-09 05:41:20,359 - BERTopic - Cluster - Completed ✓
2026-03-09 05:41:20,369 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-09 05:41:46,777 - BERTopic - Representation - Completed ✓


In [28]:
df_topic["topic"] = topics

In [29]:
df_topic.head()

,date,title_clean,clean_text,topic_text,topic
0,2024-09-22 00:00:00+00:00,"If using AI feels like a chore, try this - Boi...","Sep 18, 2024 TL;DR: Put all your AI tools in o...","If using AI feels like a chore, try this - Boi...",-1
1,2023-11-10 00:00:00+00:00,The Road Ahead: How China's AI Foundation Mode...,"By Novembro 8, 2023 O camiño por diante: como ...",The Road Ahead: How China's AI Foundation Mode...,-1
2,2023-11-19 00:00:00+00:00,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,44
3,2023-09-07 00:00:00+00:00,Zoom Expands AI Offering with AI Companion and...,By Robert AndrewRelated Post Technology Cisco ...,Zoom Expands AI Offering with AI Companion and...,-1
4,2023-08-04 00:00:00+00:00,Pro-AI Thinking: Enhancing Industrial Environm...,By Mampho Brescia Related Post News Technologi...,Pro-AI Thinking: Enhancing Industrial Environm...,57


In [30]:
topic_info = topic_model.get_topic_info()
topic_info.head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,49879,-1_prnewsfoto_npr_sap_salary,"[prnewsfoto, npr, sap, salary, workday, cdt ho...",[Sleepless AI (AI) Tops 24 Hour Trading Volume...
1,0,8088,0_amd_etf_motley_fool,"[amd, etf, motley, fool, motley fool, ses, gpu...",[Could AMD Stock Mirror Nvidia's Artificial In...
2,1,5113,1_clinical_imaging_doctors_drug discovery,"[clinical, imaging, doctors, drug discovery, d...",[Revolutionizing Cancer Diagnostics: How Multi...
3,2,3086,2_guinea_congo_islands_construction consumer,"[guinea, congo, islands, construction consumer...",[Bubblr Inc. Debuts AI Seek's Enhanced Website...
4,3,3065,3_iit_tcs_jio_crore,"[iit, tcs, jio, crore, indiaai, narendra, prad...",[Gartner Predicts 40% Failure Rate for Agentic...
5,4,2203,4_malicious_phishing_malware_vulnerability,"[malicious, phishing, malware, vulnerability, ...",[Hackers are using AI to create vicious malwar...
6,5,1871,5_lending_financial institutions_zest_mortgage,"[lending, financial institutions, zest, mortga...",[Zest AI Unveils New Post-origination Risk Ana...
7,6,1864,6_alibaba_baidu_ernie_r1,"[alibaba, baidu, ernie, r1, shanghai, ernie bo...",[Baidu to implement ChatGPT-like Ernie Bot cha...
8,7,1610,7_hinton_extinction_geoffrey hinton_geoffrey,"[hinton, extinction, geoffrey hinton, geoffrey...",[Scientists warn of AI dangers but don’t agree...
9,8,1553,8_self driving_hyundai_autonomous vehicles_vol...,"[self driving, hyundai, autonomous vehicles, v...",[Ford disbands Argo AI autonomous vehicle unit...


In [31]:
df_topic["topic"].value_counts().head(20)

,count
topic,
-1,49879
0,8088
1,5113
2,3086
3,3065
4,2203
5,1871
6,1864
7,1610


In [32]:
for topic_id in topic_info["Topic"].head(15):
    if topic_id == -1:
        continue
    print(f"\n===== Topic {topic_id} =====")
    print(topic_model.get_topic(topic_id)[:10])


===== Topic 0 =====
[('amd', np.float64(0.021630216916607072)), ('etf', np.float64(0.017310179469854026)), ('motley', np.float64(0.014138709015327478)), ('fool', np.float64(0.014015162278130166)), ('motley fool', np.float64(0.014009530074956734)), ('ses', np.float64(0.01333478417736865)), ('gpus', np.float64(0.01227842204883225)), ('gpu', np.float64(0.0121049607653876)), ('palantir', np.float64(0.011474886292377764)), ('dell', np.float64(0.011041312227878623))]

===== Topic 1 =====
[('clinical', np.float64(0.036091588954305594)), ('imaging', np.float64(0.018699401361705933)), ('doctors', np.float64(0.016884365069737995)), ('drug discovery', np.float64(0.016112585001978444)), ('diagnosis', np.float64(0.015588350289826327)), ('diseases', np.float64(0.014807492471436678)), ('breast', np.float64(0.014307015134319718)), ('diagnostic', np.float64(0.013608965203481288)), ('clinicians', np.float64(0.012066323449729393)), ('fda', np.float64(0.011753170377434046))]

===== Topic 2 =====
[('guine

In [33]:
topic_model.visualize_topics()

In [34]:
topic_model.visualize_barchart()

In [35]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,49879,-1_prnewsfoto_npr_sap_salary,"[prnewsfoto, npr, sap, salary, workday, cdt ho...",[Sleepless AI (AI) Tops 24 Hour Trading Volume...
1,0,8088,0_amd_etf_motley_fool,"[amd, etf, motley, fool, motley fool, ses, gpu...",[Could AMD Stock Mirror Nvidia's Artificial In...
2,1,5113,1_clinical_imaging_doctors_drug discovery,"[clinical, imaging, doctors, drug discovery, d...",[Revolutionizing Cancer Diagnostics: How Multi...
3,2,3086,2_guinea_congo_islands_construction consumer,"[guinea, congo, islands, construction consumer...",[Bubblr Inc. Debuts AI Seek's Enhanced Website...
4,3,3065,3_iit_tcs_jio_crore,"[iit, tcs, jio, crore, indiaai, narendra, prad...",[Gartner Predicts 40% Failure Rate for Agentic...
...,...,...,...,...,...
108,107,213,107_supply distribution_diluted_mexc_calculated,"[supply distribution, diluted, mexc, calculate...","[Koala AI (KOKO) Tokenomics: Market Insights, ..."
109,108,207,108_softbank_stargate_masayoshi_masayoshi son,"[softbank, stargate, masayoshi, masayoshi son,...",[SoftBank completes $41 billion OpenAI investm...
110,109,206,109_scraping_cloudflare_crawlers_crawler,"[scraping, cloudflare, crawlers, crawler, robo...",[News publishers back Cloudflare blocking of a...
111,110,204,110_gates_nadella_satya_satya nadella,"[gates, nadella, satya, satya nadella, agi, su...",[Bill Gates just published a 7-page letter abo...


In [36]:
topic_model.reduce_outliers(docs, topics)

100%|██████████| 50/50 [01:14<00:00,  1.48s/it]


[np.int64(39),
 np.int64(8),
 44,
 np.int64(57),
 57,
 4,
 0,
 np.int64(0),
 29,
 53,
 4,
 0,
 19,
 0,
 48,
 23,
 11,
 13,
 np.int64(11),
 48,
 np.int64(16),
 np.int64(55),
 1,
 1,
 np.int64(25),
 103,
 np.int64(42),
 83,
 np.int64(41),
 np.int64(59),
 0,
 18,
 48,
 np.int64(42),
 92,
 np.int64(5),
 np.int64(69),
 np.int64(0),
 59,
 1,
 8,
 np.int64(7),
 np.int64(15),
 110,
 4,
 32,
 31,
 0,
 24,
 3,
 90,
 2,
 2,
 2,
 2,
 2,
 2,
 47,
 np.int64(100),
 17,
 20,
 14,
 np.int64(25),
 32,
 np.int64(102),
 16,
 16,
 22,
 np.int64(110),
 np.int64(45),
 np.int64(91),
 np.int64(37),
 np.int64(14),
 np.int64(39),
 np.int64(42),
 np.int64(58),
 np.int64(77),
 np.int64(24),
 29,
 32,
 99,
 3,
 0,
 6,
 np.int64(11),
 11,
 85,
 22,
 np.int64(5),
 1,
 np.int64(62),
 np.int64(83),
 97,
 np.int64(33),
 0,
 0,
 0,
 np.int64(14),
 np.int64(19),
 3,
 60,
 np.int64(77),
 19,
 27,
 91,
 4,
 np.int64(105),
 30,
 54,
 np.int64(77),
 np.int64(43),
 33,
 np.int64(62),
 4,
 0,
 31,
 38,
 np.int64(87),
 3,
 92,
 

In [37]:
topic_info = topic_model.get_topic_info()
topic_info.to_csv(ARTIFACTS_DIR / "topic_info_full.csv", index=False)
topic_info.head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,49879,-1_prnewsfoto_npr_sap_salary,"[prnewsfoto, npr, sap, salary, workday, cdt ho...",[Sleepless AI (AI) Tops 24 Hour Trading Volume...
1,0,8088,0_amd_etf_motley_fool,"[amd, etf, motley, fool, motley fool, ses, gpu...",[Could AMD Stock Mirror Nvidia's Artificial In...
2,1,5113,1_clinical_imaging_doctors_drug discovery,"[clinical, imaging, doctors, drug discovery, d...",[Revolutionizing Cancer Diagnostics: How Multi...
3,2,3086,2_guinea_congo_islands_construction consumer,"[guinea, congo, islands, construction consumer...",[Bubblr Inc. Debuts AI Seek's Enhanced Website...
4,3,3065,3_iit_tcs_jio_crore,"[iit, tcs, jio, crore, indiaai, narendra, prad...",[Gartner Predicts 40% Failure Rate for Agentic...
5,4,2203,4_malicious_phishing_malware_vulnerability,"[malicious, phishing, malware, vulnerability, ...",[Hackers are using AI to create vicious malwar...
6,5,1871,5_lending_financial institutions_zest_mortgage,"[lending, financial institutions, zest, mortga...",[Zest AI Unveils New Post-origination Risk Ana...
7,6,1864,6_alibaba_baidu_ernie_r1,"[alibaba, baidu, ernie, r1, shanghai, ernie bo...",[Baidu to implement ChatGPT-like Ernie Bot cha...
8,7,1610,7_hinton_extinction_geoffrey hinton_geoffrey,"[hinton, extinction, geoffrey hinton, geoffrey...",[Scientists warn of AI dangers but don’t agree...
9,8,1553,8_self driving_hyundai_autonomous vehicles_vol...,"[self driving, hyundai, autonomous vehicles, v...",[Ford disbands Argo AI autonomous vehicle unit...


In [40]:
import pandas as pd

topic_labels = pd.DataFrame([

    {"topic": -1, "topic_name": "Outliers / noise", "industry_label": "Noise", "keep": 0},

    {"topic": 0, "topic_name": "AI chips and GPUs", "industry_label": "AI Infrastructure", "keep": 1},

    {"topic": 1, "topic_name": "AI in healthcare and medical imaging", "industry_label": "Healthcare", "keep": 1},

    {"topic": 2, "topic_name": "Newswire template / geography noise", "industry_label": "Noise", "keep": 0},

    {"topic": 3, "topic_name": "Enterprise AI adoption (IT services)", "industry_label": "Enterprise IT", "keep": 1},

    {"topic": 4, "topic_name": "AI cybersecurity", "industry_label": "Cybersecurity", "keep": 1},

    {"topic": 5, "topic_name": "AI in lending and financial services", "industry_label": "Finance", "keep": 1},

    {"topic": 6, "topic_name": "China AI ecosystem", "industry_label": "Global AI Competition", "keep": 0},

    {"topic": 7, "topic_name": "AI safety and existential risk", "industry_label": "AI Governance / Risk", "keep": 1},

    {"topic": 8, "topic_name": "Autonomous vehicles and mobility AI", "industry_label": "Transportation", "keep": 1},

    {"topic": 9, "topic_name": "Consumer AI assistants and devices", "industry_label": "Consumer Electronics", "keep": 1},

    {"topic": 10, "topic_name": "AI in education", "industry_label": "Education", "keep": 1},

    {"topic": 11, "topic_name": "AI and crypto / Web3", "industry_label": "Crypto / Web3", "keep": 1},

    {"topic": 12, "topic_name": "Website navigation / template noise", "industry_label": "Noise", "keep": 0},

    {"topic": 13, "topic_name": "AI regulation and national policy", "industry_label": "Government / Regulation", "keep": 1},

    {"topic": 14, "topic_name": "AI adoption in small businesses", "industry_label": "Small Business", "keep": 1},

    {"topic": 15, "topic_name": "National AI strategy and global hubs", "industry_label": "Government / Regulation", "keep": 1},

    {"topic": 16, "topic_name": "Generative AI for media and video", "industry_label": "Media / Content Creation", "keep": 1},

    {"topic": 17, "topic_name": "AI company governance events", "industry_label": "AI Governance / Risk", "keep": 1},

    {"topic": 18, "topic_name": "AI model competition and launches", "industry_label": "Noise", "keep": 0}

])

topic_labels

,topic,topic_name,industry_label,keep
0,-1,Outliers / noise,Noise,0
1,0,AI chips and GPUs,AI Infrastructure,1
2,1,AI in healthcare and medical imaging,Healthcare,1
3,2,Newswire template / geography noise,Noise,0
4,3,Enterprise AI adoption (IT services),Enterprise IT,1
5,4,AI cybersecurity,Cybersecurity,1
6,5,AI in lending and financial services,Finance,1
7,6,China AI ecosystem,Global AI Competition,0
8,7,AI safety and existential risk,AI Governance / Risk,1
9,8,Autonomous vehicles and mobility AI,Transportation,1


In [41]:
TOPIC_LABELS_PATH = ARTIFACTS_DIR / "topic_labels_manual.csv"

topic_labels.to_csv(TOPIC_LABELS_PATH, index=False)

print("Saved:", TOPIC_LABELS_PATH)

Saved: /content/drive/MyDrive/nextgen_nlp_final/artifacts/topic_labels_manual.csv


In [42]:
df_topic_labeled = df_topic.merge(topic_labels, on="topic", how="left")

df_topic_labeled.head()

,date,title_clean,clean_text,topic_text,topic,topic_name,industry_label,keep
0,2024-09-22 00:00:00+00:00,"If using AI feels like a chore, try this - Boi...","Sep 18, 2024 TL;DR: Put all your AI tools in o...","If using AI feels like a chore, try this - Boi...",-1,Outliers / noise,Noise,0.0
1,2023-11-10 00:00:00+00:00,The Road Ahead: How China's AI Foundation Mode...,"By Novembro 8, 2023 O camiño por diante: como ...",The Road Ahead: How China's AI Foundation Mode...,-1,Outliers / noise,Noise,0.0
2,2023-11-19 00:00:00+00:00,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,44,NaN,NaN,NaN
3,2023-09-07 00:00:00+00:00,Zoom Expands AI Offering with AI Companion and...,By Robert AndrewRelated Post Technology Cisco ...,Zoom Expands AI Offering with AI Companion and...,-1,Outliers / noise,Noise,0.0
4,2023-08-04 00:00:00+00:00,Pro-AI Thinking: Enhancing Industrial Environm...,By Mampho Brescia Related Post News Technologi...,Pro-AI Thinking: Enhancing Industrial Environm...,57,NaN,NaN,NaN


In [43]:
df_topic_keep = df_topic_labeled[df_topic_labeled["keep"] == 1].copy()

print("Remaining articles:", df_topic_keep.shape)

df_topic_keep["industry_label"].value_counts()

Remaining articles: (33765, 8)


,count
industry_label,
AI Infrastructure,8088
Healthcare,5113
Enterprise IT,3065
AI Governance / Risk,2627
Government / Regulation,2437
Cybersecurity,2203
Finance,1871
Transportation,1553
Consumer Electronics,1548


In [48]:
df_topic_keep.shape

(33765, 8)

In [44]:
df_ner_input = df_topic_keep[
    ["date", "title_clean", "clean_text", "topic", "topic_name", "industry_label"]
].copy()

NER_INPUT_PATH = ARTIFACTS_DIR / "ner_input_articles.parquet"

df_ner_input.to_parquet(NER_INPUT_PATH, index=False)

print("Saved:", NER_INPUT_PATH)

Saved: /content/drive/MyDrive/nextgen_nlp_final/artifacts/ner_input_articles.parquet
